# Rasenfläche + Geothermie-Machbarkeit
**Left Join für die Rasenfläche | Boolean Geothermie-Flag**

Der Wohnbauflächen-Bestätigungs-Join (welche Gebäude als „wohnbaulich" gelten)
liegt in `02_nrw_clustering.ipynb`, weil nur dieses Notebook eine saubere
Trainingsmenge fürs Clustering braucht. Dieses Notebook behandelt die Teile,
die für **jedes** Gebäude gelten, unabhängig vom Bestätigungsstatus:

1. **Left Join** — Rasenfläche für alle Gebäude berechnen
   → ohne Treffer ist die Rasenfläche 0 (nicht ausgeschlossen, nur keine Daten)
2. **Boolean Geothermie-Flag** — Rasenfläche ≥ benötigte Sondenfläche

Das Geothermie-Flag braucht die Spitzenleistung (`peak_heat_power_kw_1/2/3`, aus
`03_nrw_spitzenwaermeleistung.ipynb`, direkt in der Arbeitsdatei) und die
Kennlinie (`geothermie_kennlinie.csv`, aus
`04_nrw_geothermie_kennlinie.ipynb`). Dieses Notebook läuft deshalb **nach**
beiden — siehe README für die komplette Laufreihenfolge.

## 1. Konfiguration

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely import wkb

warnings.filterwarnings("ignore")

BASE_DIR      = "."
FILE_GEBAEUDE = os.path.join(BASE_DIR, "nrw_buildings.parquet")
FILE_WOHN     = os.path.join(BASE_DIR, "katasterdaten", "wohnflaechen_nrw.parquet")
FILE_KENNLINIE = os.path.join(BASE_DIR, "nrw_results", "geothermie_kennlinie.csv")

print("Bereit")
print(f'wohnflaechen vorhanden: {os.path.exists(FILE_WOHN)}')
print(f'Arbeitsdatei vorhanden: {os.path.exists(FILE_GEBAEUDE)}')
print(f'Kennlinie vorhanden: {os.path.exists(FILE_KENNLINIE)}')

Bereit
wohnflaechen vorhanden: True
Arbeitsdatei vorhanden: True
Kennlinie vorhanden: False


## 2. Wohnbauflächen-Datensatz laden

Der Datensatz wurde vor der Übergabe bereits auf Wohnbauflächen-Parzellen
vorgefiltert. Alle ~4,5 Mio. Parzellen sind wohnbaulich.

In [2]:
wohn = pd.read_parquet(FILE_WOHN)
print(f'Geladen: {len(wohn):,} Parzellen')
print(f'Spalten: {wohn.columns.tolist()}')
print(f'\nflaeche-Statistik: min={wohn["flaeche"].min():.0f}  '
      f'mittel={wohn["flaeche"].mean():.0f}  max={wohn["flaeche"].max():.0f} m2')

Geladen: 4,534,540 Parzellen
Spalten: ['geometry', 'flaeche', 'kreis', 'tntxt']

flaeche-Statistik: min=0  mittel=897  max=7508588 m2


In [3]:
# WKB-Geometrie dekodieren
wohn["geometry_decoded"] = wohn["geometry"].apply(
    lambda x: wkb.loads(x) if isinstance(x, bytes) else x
)
wohn_gdf = gpd.GeoDataFrame(wohn, geometry="geometry_decoded", crs="EPSG:25832")
print(f'Geometrie dekodiert: {len(wohn_gdf):,} Parzellen  CRS: EPSG:25832')
print(f'Ausdehnung: {wohn_gdf.total_bounds.round(0)}')

Geometrie dekodiert: 4,534,540 Parzellen  CRS: EPSG:25832
Ausdehnung: [ 280379. 5579055.  530488. 5819862.]


## 3. Gebäudedaten laden

Die gemeinsame Arbeitsdatei liegt in EPSG:25832 vor (bereits in 01 umprojiziert)
— passend zu den Wohnbauflächen. Die Centroide für den räumlichen Join werden
aus der `footprint`-Geometrie berechnet.

In [4]:
gebaeude = gpd.read_parquet(FILE_GEBAEUDE)
if gebaeude.geometry.name != "footprint":
    gebaeude = gebaeude.set_geometry("footprint")
gebaeude["centroid"] = gebaeude["footprint"].centroid

print(f'Gebaeude geladen: {len(gebaeude):,}')
print(f'Ausdehnung: {gebaeude.total_bounds.round(0)}')

Gebaeude geladen: 4,134,282
Ausdehnung: [ 280801. 5578390.  530435. 5819749.]


## 4. Nicht zugeordnete Gebäude untersuchen

Schnelle Prüfung der Gebäude, deren Zentroid außerhalb jeder
Wohnbauflächen-Parzelle liegt. Diese Gebäude bleiben im Datensatz (nichts wird
hier ausgeschlossen) — sie bekommen in Abschnitt 5 einfach `lawn_area = 0`,
weil es keine Parzelle gibt, aus der sich eine Rasenfläche berechnen ließe.

In [5]:
# Schneller Join, um nicht zugeordnete Gebaeude zu identifizieren
print("Schneller Join zur Identifikation nicht zugeordneter Gebaeude...")
quick_match = gpd.sjoin(gebaeude.set_geometry("centroid"), wohn_gdf,
                        how="left", predicate="intersects", rsuffix="lika")
unmatched_mask = quick_match["index_lika"].isna()
unmatched_ids  = quick_match[unmatched_mask]["id"].values
print(f'Nicht zugeordnet: {len(unmatched_ids):,}  '
      f'({len(unmatched_ids)/len(gebaeude)*100:.1f}%)')

Schneller Join zur Identifikation nicht zugeordneter Gebaeude...
Nicht zugeordnet: 427,596  (10.3%)


## 5. Rasenfläche per Left Join

Alle Gebäude bekommen einen Rasenflächen-Wert. Ohne Treffer → Rasenfläche = 0
(kein zugehöriges Grundstück im Datensatz). Das ist leichter zu erklären als
negative Verhältnisse aus fehlerhaften Puffer-Treffern.

In [6]:
def berechne_rasenflaeche(matched_gdf):
    # Rasenflaeche = Parzellenflaeche minus Grundflaeche aller Gebaeude darauf,
    # gleichmaessig auf diese Gebaeude verteilt.
    grouped = matched_gdf.groupby("index_lika")
    rasen_df = grouped.agg(
        gesamt_osm_flaeche=("area_m2", "sum"),
        lika_flaeche=("flaeche", "first"),
        anzahl_osm_gebaeude=("area_m2", "count")
    )
    rasen_df["lawn_area"] = (
        rasen_df["lika_flaeche"] - rasen_df["gesamt_osm_flaeche"]
    ) / rasen_df["anzahl_osm_gebaeude"]
    return rasen_df["lawn_area"]

print("Left Join fuer die Rasenflaeche (alle Gebaeude)...")

# Left Join — alle Gebaeude, Rasenflaeche = 0 ohne Treffer
left = gpd.sjoin(
    gebaeude.set_geometry("centroid"), wohn_gdf,
    how="left", predicate="intersects", rsuffix="lika"
).set_geometry("footprint")

# Wohnbauflaechen sind disjunkt — hoechstens ein Treffer je Zentroid.
# Falls ein Zentroid doch mehrfach trifft, den ersten behalten, damit die
# Zeilenzahl der Arbeitsdatei stabil bleibt. Der urspruengliche Index bleibt
# erhalten (kein merge, keine Neuindizierung) — die Zuweisung zurueck in
# gebaeude erfolgt spaeter indexbasiert.
left = left[~left.index.duplicated(keep="first")]

# Rasenflaeche je Parzelle berechnen und per Parzellen-Index (index_lika)
# auf die Gebaeude mappen — indexsicher, ohne merge-Reset.
rasen_je_parzelle = berechne_rasenflaeche(left.dropna(subset=["index_lika"]))
left["lawn_area"] = left["index_lika"].map(rasen_je_parzelle)

# Ohne Treffer -> Rasenflaeche = 0
left["lawn_area"] = left["lawn_area"].fillna(0).clip(lower=0)

print(f'  Alle Gebaeude          : {len(left):,}')
print(f'  Mit Rasenflaeche > 0   : {(left["lawn_area"] > 0).sum():,}')
print(f'  Mit Rasenflaeche = 0   : {(left["lawn_area"] == 0).sum():,}'
      '  (kein Parzellen-Treffer)')
rasen = left["lawn_area"]
print(f'\nRasenflaeche-Statistik (alle Gebaeude inkl. 0):')
print(f'  mittel  : {rasen.mean():.1f} m2')
print(f'  median  : {rasen.median():.1f} m2')
print(f'  max     : {rasen.max():.1f} m2')

# Rasenflaeche indexbasiert zurueck in das Arbeits-DataFrame
gebaeude["lawn_area"] = left["lawn_area"]
gebaeude["lawn_area"] = gebaeude["lawn_area"].fillna(0)

Left Join fuer die Rasenflaeche (alle Gebaeude)...
  Alle Gebaeude          : 4,134,282
  Mit Rasenflaeche > 0   : 3,699,589
  Mit Rasenflaeche = 0   : 434,693  (kein Parzellen-Treffer)

Rasenflaeche-Statistik (alle Gebaeude inkl. 0):
  mittel  : 654.9 m2
  median  : 360.5 m2
  max     : 4867696.3 m2


## 6. Boolean Geothermie-Flag

- Rasenfläche > 0 und Rasenfläche ≥ benötigte Fläche → True
- Rasenfläche = 0 (keine Parzelle) → False (Daten unzureichend)
- Rasenfläche < benötigte Fläche → False

Die Spitzenleistung (`peak_heat_power_kw_1/2/3`) kommt direkt aus der
Arbeitsdatei (Abschnitt 5 von 03), die benötigte Fläche aus der Kennlinie von 04.

In [7]:
if not os.path.exists(FILE_KENNLINIE):
    print("geothermie_kennlinie.csv nicht gefunden — "
          "zuerst 04_nrw_geothermie_kennlinie.ipynb ausfuehren")
else:
    kennlinie = pd.read_csv(FILE_KENNLINIE)

    # Vektorisierte Kennlinien-Abfrage per searchsorted
    kennlinie_sortiert = kennlinie.sort_values("heat_power_kw").reset_index(drop=True)
    leistung_werte = kennlinie_sortiert["heat_power_kw"].values
    flaeche_werte = kennlinie_sortiert["flaeche_m2"].values

    for szenario in [1, 2, 3]:
        spitz = gebaeude[f"peak_heat_power_kw_{szenario}"].fillna(0).values
        indizes = np.searchsorted(leistung_werte, spitz)
        indizes = np.clip(indizes, 0, len(flaeche_werte) - 1)
        gebaeude[f"required_area_m2_{szenario}"] = flaeche_werte[indizes].astype(float)
        gebaeude.loc[gebaeude[f"peak_heat_power_kw_{szenario}"].isna(),
                     f"required_area_m2_{szenario}"] = np.nan

    # Boolean-Flag je Szenario
    for szenario in [1, 2, 3]:
        gebaeude[f"geothermal_feasible_{szenario}"] = (
            (gebaeude["lawn_area"] > 0) &
            (gebaeude["lawn_area"] >= gebaeude[f"required_area_m2_{szenario}"])
        )

    n_machbar = gebaeude["geothermal_feasible_1"].sum()
    print(f'Geothermie-Machbarkeit (alle {len(gebaeude):,} Gebaeude):')
    print(f'  Machbar (True) : {n_machbar:,}  ({n_machbar/len(gebaeude)*100:.1f}%)')
    print(f'  Nicht machbar  : {len(gebaeude)-n_machbar:,}')
    print()
    print("Nach Groessenklasse:")
    for sc in ["SFH", "TH", "MFH", "AB"]:
        sub = gebaeude[gebaeude["size_class"] == sc]
        if sub.empty:
            continue
        n_f = sub["geothermal_feasible_1"].sum()
        print(f'  {sc}: {n_f:>8,} machbar von {len(sub):,}  ({n_f/len(sub)*100:.1f}%)')

geothermie_kennlinie.csv nicht gefunden — zuerst 04_nrw_geothermie_kennlinie.ipynb ausfuehren


## 7. In die gemeinsame Arbeitsdatei schreiben

Rasenfläche, benötigte Fläche und Geothermie-Flag werden für jedes Gebäude in
`nrw_buildings.parquet` zurückgeschrieben. Die Bestätigung „wohnbaulich" selbst
erfolgt in `02_nrw_clustering.ipynb`.

In [8]:
neue_spalten = ["lawn_area", "required_area_m2_1", "required_area_m2_2",
                "required_area_m2_3",
                "geothermal_feasible_1", "geothermal_feasible_2",
                "geothermal_feasible_3"]
vorhanden = [c for c in neue_spalten if c in gebaeude.columns]
gebaeude[vorhanden] = gebaeude[vorhanden].round(2)

# centroide-Spalte (Arbeitshilfe aus Abschnitt 3) nicht mitschreiben
gebaeude = gebaeude.drop(columns=[c for c in ["centroid"] if c in gebaeude.columns])

gebaeude.to_parquet(FILE_GEBAEUDE, index=False)
print(f"ueberschrieben: {FILE_GEBAEUDE}")
print(f'Zeilen: {len(gebaeude):,}  Spalten: {len(gebaeude.columns)}')
print("\nFertig.")

ueberschrieben: ./nrw_buildings.parquet
Zeilen: 4,134,282  Spalten: 43

Fertig.
